In [3]:
import subprocess
subprocess.run(['pip', 'install', 'findspark'])

CompletedProcess(args=['pip', 'install', 'findspark'], returncode=0)

In [1]:
import findspark
findspark.init(r'C:\spark')

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CustomerBehaviorStream") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.8") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 3.5.8


In [1]:
import findspark
findspark.init(r'C:\spark')

import os
os.environ['SPARK_LOCAL_DIRS'] = r'C:\Users\USER PC\bigdata_project\spark-tmp'
os.environ['HADOOP_HOME'] = r'C:\hadoop-3.3.6'

# Create temp dir if it doesn't exist
os.makedirs(r'C:\Users\USER PC\bigdata_project\spark-tmp', exist_ok=True)

from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("CustomerBehaviorStream") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.8") \
    .config("spark.local.dir", r"C:\Users\USER PC\bigdata_project\spark-tmp") \
    .config("spark.sql.warehouse.dir", r"C:\Users\USER PC\bigdata_project\spark-warehouse") \
    .config("spark.hadoop.fs.defaultFS", "file:///") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

schema = StructType([
    StructField("customer_id", StringType()),
    StructField("age", StringType()),
    StructField("session_duration_mins", StringType()),
    StructField("pages_visited", StringType()),
    StructField("clicks", StringType()),
    StructField("time_on_site_mins", StringType()),
    StructField("previous_purchases", StringType()),
    StructField("cart_additions", StringType()),
    StructField("device_type", StringType()),
    StructField("browser", StringType()),
    StructField("region", StringType()),
    StructField("membership_level", StringType()),
    StructField("discount_applied", StringType()),
    StructField("time_since_last_visit_days", StringType()),
    StructField("will_purchase", StringType())
])

df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "customer-behavior") \
    .option("startingOffsets", "earliest") \
    .load()

parsed = df.select(from_json(col("value").cast("string"), schema).alias("data")).select("data.*")

query = parsed.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", False) \
    .option("numRows", 5) \
    .trigger(processingTime="10 seconds") \
    .start()

print("PySpark Streaming started...")
query.awaitTermination(60)

PySpark Streaming started...


False